In [ ]:
#| hide
#| default_exp process

# process
> A notebook processor
- order: 3

In [ ]:
#| export
from nbdev.config import *
from nbdev.maker import *
from nbdev.imports import *

from fastcore.nbio import *
from fastcore.nbio import _meta_directives
from fastcore.script import *
from fastcore.imports import *

from collections import defaultdict

In [ ]:
#| hide
from fastcore.test import *
from pdb import set_trace
from importlib import reload
from fastcore import shutil

Special comments at the start of a cell provide information to `nbdev` about how to process it. The parsing primitives live in `fastcore.nbio` (the `langs` table, `nb_lang`, `first_code_ln`, and `NbCell`'s `directives` property and `remove_directives` method) and are available here via its re-exports; this module builds the notebook processing pipeline on top of them.

In [ ]:
#| export
def opt_set(var, newval):
    "newval if newval else var"
    return newval if newval else var

In [ ]:
#| export
def instantiate(x, **kwargs):
    "Instantiate `x` if it's a type"
    return x(**kwargs) if isinstance(x,type) else x

def _mk_procs(procs, nb): return L(procs).map(instantiate, nb=nb)

In [ ]:
#| export
def _is_direc(f): return getattr(f, '__name__', '-')[-1]=='_'

In [ ]:
#| export
class NBProcessor:
    "Process cells and nbdev comments in a notebook"
    def __init__(self, path=None, procs=None, nb=None, debug=False, rm_directives=True, process=False):
        self.nb = read_nb(path) if nb is None else nb
        self.lang = nb_lang(self.nb)
        for cell in self.nb.cells: cell.directives_ = cell.directives
        nbdirs = {k:v for k,v in _meta_directives(self.nb.get('metadata')).items() if not any(k in c.directives_ for c in self.nb.cells)}
        if nbdirs:
            fc = first(c for c in self.nb.cells if c.cell_type=='code')
            if fc is not None: fc.directives_ = nbdirs | fc.directives_
        if rm_directives:
            for cell in self.nb.cells: cell.remove_directives(quarto=rm_directives=='quarto')
        self.procs = _mk_procs(procs, nb=self.nb)
        self.debug,self.rm_directives = debug,rm_directives
        if process: self.process()

    def _process_cell(self, proc, cell):
        if not hasattr(cell,'source'): return
        if cell.cell_type=='code' and cell.directives_:
            # Option 1: `proc` is directive name with `_` suffix
            f = getattr(proc, '__name__', '-').rstrip('_')
            if f in cell.directives_: self._process_comment(proc, cell, f)
            
            # Option 2: `proc` contains a method named `_{directive}_`
            for cmd in cell.directives_:
                f = getattr(proc, f'_{cmd}_', None)
                if f: self._process_comment(f, cell, cmd)
        if callable(proc) and not _is_direc(proc): cell = opt_set(cell, proc(cell))

    def _process_comment(self, proc, cell, cmd):
        args = cell.directives_[cmd].split()
        if self.debug: print(cmd, args, proc)
        return proc(cell, *args)
        
    def _proc(self, proc):
        if hasattr(proc,'begin'): proc.begin()
        for cell in self.nb.cells: self._process_cell(proc, cell)
        if hasattr(proc,'end'): proc.end()
        self.nb.cells = [c for c in self.nb.cells if c and getattr(c,'source',None) is not None]
        for i,cell in enumerate(self.nb.cells): cell.idx_ = i

    def process(self):
        "Process all cells with all processors"
        for proc in self.procs: self._proc(proc)

Cell processors can be callables (e.g regular functions), in which case they are called for every cell (set a cell's source to `None` to remove the cell):

In [ ]:
everything_fn = '../../tests/01_everything.ipynb'

def print_execs(cell):
    if 'exec' in cell.source: print(cell.source)

NBProcessor(everything_fn, print_execs).process()

---
title: Foo
execute:
  echo: false
---
exec("o_y=1")
exec("p_y=1")
_all_ = [o_y, 'p_y']


Directives are put in a cell attribute `directives_` as a dictionary keyed by directive name, with the raw value string as each value (`''` for a bare directive). When a processor method receives a directive, the value is whitespace-split into positional arguments:

Notebook-level directives can be stored in the notebook metadata, as a dict under an `nbdev` key: they are merged into the first code cell's `directives_`, so notebook-scope directives like `default_exp` need no cell to live in. A directive of the same name anywhere in a cell takes priority.

In [ ]:
_nbm = dict2nb(dict(cells=[mk_cell('# a note', 'markdown'), mk_cell('1+1')],
                    metadata=dict(nbdev=dict(default_exp='core')), nbformat=4, nbformat_minor=5))
NBProcessor(nb=_nbm)
test_eq(_nbm.cells[1].directives_, {'default_exp': 'core'})

_nbm = dict2nb(dict(cells=[mk_cell('#| default_exp: other\n1+1')],
                    metadata=dict(nbdev=dict(default_exp='core')), nbformat=4, nbformat_minor=5))
NBProcessor(nb=_nbm)
test_eq(_nbm.cells[0].directives_, {'default_exp': 'other'})

In [ ]:
def printme_func(cell):
    if cell.directives_ and 'printme' in cell.directives_: print(cell.directives_['printme'])

NBProcessor(everything_fn, printme_func).process()

testing


However, a more convenient way to handle comment directives is to use a *class* as a processor, and include a method in your class with the same name as your directive, surrounded by underscores:

In [ ]:
class _PrintExample:
    def _printme_(self, cell, to_print): print(to_print)

NBProcessor(everything_fn, _PrintExample()).process()

testing


In the case that your processor supports just one comment directive, you can just use a regular function, with the same name as your directive, but with an underscore appended -- here `printme_` is identical to `_PrintExample` above:

In [ ]:
def printme_(cell, to_print): print(to_print)

NBProcessor(everything_fn, printme_).process()

testing


In [ ]:
NBProcessor(everything_fn, _PrintExample()).process()

testing


In [ ]:
#| export
class Processor:
    "Base class for processors"
    def __init__(self, nb): self.nb = nb
    def cell(self, cell): pass
    def __call__(self, cell): return self.cell(cell)

For more complex behavior, inherit from `Processor`, and override one of more of `begin()` (called before any cells are processed), `cell()` (called for each cell), and `end()` (called after all cells are processed). You can also include comment directives (such as the `_printme` example above) in these subclasses. Subclasses will automatically have access to `self.nb`, containing the processed notebook.

In [ ]:
class CountCellProcessor(Processor):
    def begin(self):
        print(f"First cell:\n{self.nb.cells[0].source}")
        self.count=0
    def cell(self, cell):
        if cell.cell_type=='code': self.count += 1
    def end(self): print(f"* There were {self.count} code cells")

In [ ]:
NBProcessor(everything_fn, CountCellProcessor).process()

First cell:
---
title: Foo
execute:
  echo: false
---
* There were 26 code cells


## Export -

In [ ]:
#| hide
from nbdev.maker import _basic_export_nb2

In [ ]:
#| eval: false
#| hide
_basic_export_nb2('01_config.ipynb', 'read')
_basic_export_nb2('02_maker.ipynb', 'maker')
_basic_export_nb2('03_process.ipynb', 'process')

g = exec_new('import nbdev.process')
assert hasattr(g['nbdev'].process, 'NBProcessor')